# Epistemological Signature: Corpus Exploration

This notebook explores the conversation corpus to understand its structure and prepare for labeling.

**Goal:** Extract patterns of inquiry, communication style, and epistemic moves from Claude conversations.

In [ ]:
# Setup
import sys
from pathlib import Path

# Add lib to path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from lib import parser, db, sampler

# Paths
PROJECT_DIR = Path.cwd().parent
CORPUS_DIR = PROJECT_DIR / 'corpus'
DB_PATH = PROJECT_DIR / 'corpus.db'
METADATA_PATH = PROJECT_DIR / 'metadata.json'

print(f"Project directory: {PROJECT_DIR}")
print(f"Corpus directory: {CORPUS_DIR}")
print(f"Database: {DB_PATH}")

## Initialize Database

Create the SQLite database and import metadata from JSON.

In [ ]:
# Initialize database schema
db.init_db(DB_PATH)
print("Database initialized")

# Import metadata
if METADATA_PATH.exists():
    count = db.import_metadata(METADATA_PATH, DB_PATH)
    print(f"Imported {count} files into database")
else:
    print("No metadata.json found - run extract_metadata.py first")

## Corpus Overview

In [ ]:
# Load into pandas for analysis
import sqlite3

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM files", conn)
conn.close()

print(f"Total conversations: {len(df)}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nAgent conversations: {df['is_agent'].sum()} ({100*df['is_agent'].mean():.1f}%)")
print(f"With reflective language: {df['has_reflective_language'].sum()} ({100*df['has_reflective_language'].mean():.1f}%)")
print(f"With command expansions: {df['has_command_expansion'].sum()} ({100*df['has_command_expansion'].mean():.1f}%)")
print(f"\nTotal user words: {df['user_word_count'].sum():,}")
print(f"Total Claude words: {df['claude_word_count'].sum():,}")

## Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Substantive user turns
ax = axes[0, 0]
df['substantive_user_turns'].clip(upper=15).hist(bins=15, ax=ax, edgecolor='black')
ax.set_xlabel('Substantive User Turns')
ax.set_ylabel('Count')
ax.set_title('Distribution of Substantive User Turns')

# User word count
ax = axes[0, 1]
df['user_word_count'].clip(upper=5000).hist(bins=30, ax=ax, edgecolor='black')
ax.set_xlabel('User Word Count')
ax.set_ylabel('Count')
ax.set_title('Distribution of User Word Count')

# Dialogue density
ax = axes[1, 0]
df[df['dialogue_density'] > 0]['dialogue_density'].clip(upper=300).hist(bins=30, ax=ax, edgecolor='black')
ax.set_xlabel('Words per User Turn')
ax.set_ylabel('Count')
ax.set_title('Dialogue Density')

# By date
ax = axes[1, 1]
df.groupby('date').size().plot(kind='bar', ax=ax)
ax.set_xlabel('Date')
ax.set_ylabel('Conversations')
ax.set_title('Conversations by Date')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

## High-Signal Candidates

Files most likely to contain rich epistemic content.

In [ ]:
# Top candidates: reflective + high engagement
top = df[df['has_reflective_language'] == True].nlargest(15, 'user_word_count')[
    ['filename', 'date', 'substantive_user_turns', 'user_word_count', 'dialogue_density']
]
top

## Labeling Progress

In [ ]:
progress = sampler.get_labeling_progress(DB_PATH)
print(f"Total files: {progress['total']}")
print(f"Labeled: {progress['labeled']} ({progress['progress_pct']}%)")
print(f"  - Rich: {progress['rich']}")
print(f"  - Not rich: {progress['not_rich']}")
print(f"Remaining: {progress['unlabeled']}")

---

## Sample for Labeling

Pull samples to review and label.

In [ ]:
# Get top candidates for labeling
candidates = sampler.sample_top_candidates(n=5, db_path=DB_PATH)

for c in candidates:
    print(f"\n{'='*60}")
    print(f"File: {c['filename']}")
    print(f"Date: {c['date']} | Turns: {c['substantive_user_turns']} | Words: {c['user_word_count']}")
    print(f"Reflective: {c['has_reflective_language']} | Density: {c['dialogue_density']}")

## View a Conversation

Parse and display a specific conversation for review.

In [ ]:
def view_conversation(filename: str, max_turns: int = 10):
    """Display a conversation for review."""
    filepath = CORPUS_DIR / filename
    if not filepath.exists():
        print(f"File not found: {filename}")
        return None
    
    conv = parser.parse_conversation(filepath)
    
    print(f"\n{'='*70}")
    print(f"FILE: {conv.filename}")
    print(f"Date: {conv.date} | Session: {conv.session_id[:20]}...")
    print(f"Turns: {len(conv.turns)} | User words: {conv.total_user_words}")
    print(f"{'='*70}\n")
    
    shown = 0
    for turn in conv.turns:
        if not turn.has_substantive_content:
            continue
        
        role_label = "👤 USER" if turn.role == 'user' else "🤖 CLAUDE"
        content = turn.content[:500] + "..." if len(turn.content) > 500 else turn.content
        
        print(f"\n--- {role_label} (turn {turn.index}, {turn.word_count} words) ---")
        print(content)
        
        shown += 1
        if shown >= max_turns:
            remaining = len([t for t in conv.turns[turn.index:] if t.has_substantive_content])
            if remaining > 0:
                print(f"\n... [{remaining} more substantive turns] ...")
            break
    
    return conv

In [ ]:
# View a specific conversation (replace with filename from candidates above)
# conv = view_conversation("claude-conversation-2025-12-26-89ce7bdd.md")

## Add Label

After reviewing, add a label to the database.

In [ ]:
def label_file(filename: str, is_rich: bool, notes: str = ""):
    """Add a label for a file."""
    file_info = db.get_file(filename, DB_PATH)
    if not file_info:
        print(f"File not found in database: {filename}")
        return
    
    label_id = db.add_label(
        file_id=file_info['id'],
        is_rich=is_rich,
        notes=notes,
        db_path=DB_PATH
    )
    
    status = "RICH" if is_rich else "NOT RICH"
    print(f"Labeled {filename} as {status}")
    if notes:
        print(f"Notes: {notes}")
    
    return label_id

In [ ]:
# Example: label a file
# label_file("claude-conversation-2025-12-26-89ce7bdd.md", is_rich=True, notes="Strong meta-reflection, conceptual exploration")

---

## Markers (Emergent Vocabulary)

As patterns emerge from labeling, capture them as markers.

In [ ]:
def add_marker(name: str, description: str = ""):
    """Add a new marker to the vocabulary."""
    marker_id = db.add_marker(name, description, DB_PATH)
    print(f"Added marker: {name}")
    return marker_id

def list_markers():
    """List all markers."""
    markers = db.get_markers(DB_PATH)
    if not markers:
        print("No markers yet.")
        return
    
    print("\nMarkers:")
    for m in markers:
        print(f"  [{m['id']}] {m['name']}: {m['description']}")

In [ ]:
# Example: add markers as they emerge
# add_marker("meta-reflection", "Conversation becomes subject of itself")
# add_marker("reframing", "User reshapes Claude's output into new question")
# add_marker("holding-ambiguity", "User maintains multiple possibilities without collapsing")

In [ ]:
list_markers()